In [ ]:
!pip install pandas playwright beautifulsoup4
get_ipython().system('playwright install chromium')

In [ ]:
import os
import csv
import uuid
import asyncio
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

In [ ]:
!apt-get update && apt-get install -y libatk1.0-0 libgdk-pixbuf-2.0-0 libgtk-3-0 libnss3

In [ ]:
!apt-get update && apt-get install -y libatk-1.0-0 libgdk-pixbuf2.0-0 libgtk-3-0 libnss3

In [ ]:
# 1. Complete Master Source List
SOURCES = [
    # --- GROUP A & B: ESTABLISHED & MID-MARKET (WEB & PDF) ---
    {"provider_name": "Tala", "legal_name": "Inventure Mobile Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://tala.co.ke/loc-terms-and-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Tala", "legal_name": "Inventure Mobile Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://tala.co.ke/privacy-policy-ke/", "raw_text": None, "local_file": None},

    {"provider_name": "Branch", "legal_name": "Branch International", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://www.branch.co/ke/legal/terms-of-use", "raw_text": None, "local_file": None},
    {"provider_name": "Branch", "legal_name": "Branch International", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.branch.co/ke/legal/privacy-policies", "raw_text": None, "local_file": None},

    {"provider_name": "Zenka", "legal_name": "Zenka Digital Limited", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://zenka.co.ke/zenka-finance-limited-terms-of-use/", "raw_text": None, "local_file": None},
    {"provider_name": "Zenka", "legal_name": "Zenka Digital Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://zenka.co.ke/zenka-finance-limited-terms-and-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Zenka", "legal_name": "Zenka Digital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://zenka.co.ke/zenka-finance-limited-privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Lendplus", "legal_name": "Aventus Technology Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://lendplus.ke/terms-and-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Lendplus", "legal_name": "Aventus Technology Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://lendplus.ke/privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Berry", "legal_name": "Finberry Capital Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://berry.ke/terms-conditions-new/", "raw_text": None, "local_file": None},
    {"provider_name": "Berry", "legal_name": "Finberry Capital Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://berry.ke/privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Mkopa", "legal_name": "M-Kopa Loan Kenya Ltd", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://www.m-kopa.com/kenya/terms-of-service", "raw_text": None, "local_file": None},
    {"provider_name": "Mkopa", "legal_name": "M-Kopa Loan Kenya Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.m-kopa.com/our-privacy-notice-and-you", "raw_text": None, "local_file": None},

    {"provider_name": "Zanifu", "legal_name": "Zanifu Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.zanifu.com/terms-and-conditions/", "raw_text": None, "local_file": None},

    {"provider_name": "Asante", "legal_name": "Asante FS East Africa Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://asante.co.ke/terms", "raw_text": None, "local_file": None},
    {"provider_name": "Asante", "legal_name": "Asante FS East Africa Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://asante.co.ke/privacy", "raw_text": None, "local_file": None},

    {"provider_name": "Jumo", "legal_name": "Jumo Kenya Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://jumo.world/privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Littlepesa", "legal_name": "Little Pesa Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://littlepesa.com/terms-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Littlepesa", "legal_name": "Little Pesa Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://littlepesa.com/privacy-policy-2/", "raw_text": None, "local_file": None},

    {"provider_name": "Pezesha", "legal_name": "Pezesha Africa Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://pezesha.com/privacy/", "raw_text": None, "local_file": None},

    {"provider_name": "4G Capital", "legal_name": "Fourth Generation Capital Limited", "document_type": "terms_and_conditions", "source_type": "official_pdf", "source_url": "Offline", "raw_text": "[PDF OFFLINE]", "local_file": "4g_capital_terms.pdf"},
    {"provider_name": "Ngao Credit", "legal_name": "Ngao Credit Limited", "document_type": "terms_and_conditions", "source_type": "official_pdf", "source_url": "Offline", "raw_text": "[PDF OFFLINE]", "local_file": "ngao_credit_terms.pdf"},
    {"provider_name": "Longitude Finance", "legal_name": "Longitude Capital Limited", "document_type": "terms_and_conditions", "source_type": "official_pdf", "source_url": "Offline", "raw_text": "[PDF OFFLINE]", "local_file": "longitude_finance_terms.pdf"},
    {"provider_name": "Getcash Capital", "legal_name": "Getcash Capital Limited", "document_type": "terms_and_conditions", "source_type": "official_pdf", "source_url": "Offline", "raw_text": "[PDF OFFLINE]", "local_file": "getcash_capital_terms.pdf"},

    {"provider_name": "Betapesa", "legal_name": "Betapesa", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://betapesa.com/asset/Beta_Privacy%20policy_20260703.html", "raw_text": None, "local_file": None},
    {"provider_name": "Helapesa", "legal_name": "Sokohela Company", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://resource.helapesa.co.ke/data-privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Spectrum Credit", "legal_name": "Spectrum Credit Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.spectrumcredit.co.ke/terms-conditions", "raw_text": None, "local_file": None},
    {"provider_name": "Spectrum Credit", "legal_name": "Spectrum Credit Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.spectrumcredit.co.ke/privacy-policy", "raw_text": None, "local_file": None},
    {"provider_name": "Spectrum Credit", "legal_name": "Spectrum Credit Ltd", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://www.spectrumcredit.co.ke/terms-of-use", "raw_text": None, "local_file": None},

    {"provider_name": "Moneza", "legal_name": "Moneza Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://britziazan.asia/privacy.html", "raw_text": None, "local_file": None},

    {"provider_name": "Malicash", "legal_name": "Malicash Investment Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://malicash.co.ke/privacy-policy/", "raw_text": None, "local_file": None},
    {"provider_name": "Malicash", "legal_name": "Malicash Investment Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://malicash.co.ke/terms-conditions/", "raw_text": None, "local_file": None},

    {"provider_name": "Lockbx", "legal_name": "Lockbx Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://lockbx.io/terms-and-conditions.html", "raw_text": None, "local_file": None},
    {"provider_name": "Lockbx", "legal_name": "Lockbx Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://lockbx.io/privacy-policy.html", "raw_text": None, "local_file": None},

    {"provider_name": "Loanplus", "legal_name": "Loan Plus Digital Credit Provider", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.loanplus.co.ke/loan_plus/android_privacy/", "raw_text": None, "local_file": None},

    {"provider_name": "Kikwetu Credit", "legal_name": "Kikwetu Credit Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://kikwetucredit.co.ke/privacy-policy/", "raw_text": None, "local_file": None},
    {"provider_name": "Kikwetu Credit", "legal_name": "Kikwetu Credit Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://kikwetucredit.co.ke/terms-and-conditions/", "raw_text": None, "local_file": None},

    {"provider_name": "iPesa", "legal_name": "iPesa", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://ipesa.top/privacy-policy", "raw_text": None, "local_file": None},
    {"provider_name": "Chapeo", "legal_name": "Chapeo Capital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://chapeo.co.ke/privacy/", "raw_text": None, "local_file": None},
    {"provider_name": "Autocheck", "legal_name": "Autocheck Kenya", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.autochek.africa/ke/privacy-policy", "raw_text": None, "local_file": None},

    # --- GROUP C: PLAY STORE TEXT INJECTIONS & WEB HYBRIDS ---
    {"provider_name": "Meta Loan", "legal_name": "Zillions Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://app-api.metaloan.co.ke/wap/agreement?name=Privacy_Policy&system=keMetaLoan", "raw_text": None, "local_file": None},
    {"provider_name": "Meta Loan", "legal_name": "Zillions Credit Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Meta Loan provide mobile loan service for Kenya customers.
• Loan Amount: KSh 1000 – KSh 80,000
• Tenure: 91 days - 365 days
• Maximum loan limit: Ksh 80,000
• maximum APR: 48% per year
For example:for loans of ksh 20,000 with a term during 91 days, the fees charged are as follows:
Monthly interest rate ： 48% / 12 = 4%
Monthly interest ： ksh 20,000 * 4% = ksh800
Total monthly payment ：ksh20,000 / 3 + ksh 800 = ksh7467
total repayment bill including principle and interest shall be:
ksh 20,000.00 + ksh20,000* 4%*3= ksh 22400

1.Why loan from Meta Loan?
• No Credit report required
• Paperless and digital process on your mobile
• No collateral requirements
• Get 7/24 access to loan at anytime and anywhere
• Disburse to your M-PESA directly in 1 minute once approved
• Available across Kenya
• As the credit score grows, the loan limit increases

2.who are eligible?
• Kenya Resident
• 18-60 years old
• has a source of income and have ability to repay

3.How to repay the loan?
• through our M-pesa Paybill 4023507 and the account no. is the mpesa phone number you sign up with/ through our APP M-pesa express
• enter the amount you pay and enter the pin of M-pesa
• timely repayment will increase your credit score, you will have more chances to get more limits

4.Security
Your data is safe with us. It is transferred over a secure HTTPS connection to us, and we do not share it with anyone without your consent except the Lenders. All the transactions are secured via a 128-bit SSL encryption.you can review our private policy before register.

5.CONTACT US
If you have any feedback, questions, or concerns, please e-mail us at help@metaloan.co.ke or in our application online service.
Address: 503，Blue Violet Plaze, Nairobi City, Kenya"""},

    {"provider_name": "Tuma Loan", "legal_name": "Futuris Tech Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://futurisltd.com/privacy.html", "raw_text": None, "local_file": None},
    {"provider_name": "Tuma Loan", "legal_name": "Futuris Tech Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Tuma Loan - a product under FUTURIS TECH LIMITED, Fast, Reliable, and Transparent Loans in Kenya
As a trusted Kenya loan app for instant cash solutions, we provide flexible loan services designed to fit the fast-paced lifestyles of people across Kenya. Powered by an intelligent and fully secure lending system, you can complete your loan application in just 3 minutes, with funds sent straight to your M-PESA account — no complicated paperwork required.

Why Choose Tuma Loan?
✅ Quick Application: Apply in just 3 minutes, 100% online
✅ Instant Disbursement: Funds sent immediately to your M-PESA account
✅ Transparent Terms: No hidden fees or conditions
✅ Available 24/7: Apply anytime, anywhere
✅ Safe & Secure: We comply with Kenya’s Data Protection Act

Product Details
- Loan Amount: Ksh 1,000 - Ksh 60,000
- Loan Term: 91 - 360 days
- Annual Percentage Rate (APR): 18% - 36%
Example Loan Calculation:
For a loan of Ksh 10,000 at 25% APR for 180 days:
Interest: Ksh 10,000 * 25% / 365 * 180 = Ksh 1,232
Transfer Fee: Ksh 20
Total Repayment: Ksh 11,252

Eligibility Criteria
- Kenyan citizen aged 18 to 60
- Valid M-PESA account
- Valid National ID
- Stable job or reliable income source

How to Apply
- Download the Tuma Loan app
- Complete the simple application form
- Receive approval in minutes
- Get funds instantly in your M-PESA account

Features You'll Love
📱 Apply anytime, anywhere - no paperwork, no face-to-face meetings
🔐 Safe and confidential application process
🔄 Flexible repayments, both online and offline
💳 No need to worry about reminders - we'll notify you in time

Privacy and Data Protection
Your privacy is our priority.
Tuma Loan uses advanced encryption and fully complies with Kenya's Data Protection Act.
We never sell your data. Instead, we use it to improve your loan experience, offering personalized insights and matching you with the best offers for your financial profile.
Read our Privacy Policy: https://futurisltd.com/privacy.html

Need Help? Contact Us Anytime
📧 Email: support@tumaloans.com
📞 Phone: +254 768 632 428
🌐 Website: www.futurisltd.com
🏢 Address: Old Mutual House, Kimathi Street, CBD, Nairobi
Tuma Loan - Empowering you financially, 24/7."""},

    {"provider_name": "ZK Pesa", "legal_name": "Chapeo Capital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://web.zkpesa.com/privacy/", "raw_text": None, "local_file": None},
    {"provider_name": "ZK Pesa", "legal_name": "Chapeo Capital Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """ZK Pesa offers quick and hassle-free cash loans tailored to meet the needs of Kenya's dynamic market. With our streamlined process, borrowers can access funds swiftly to address their financial requirements. Simply provide your M-Pesa account details, and enjoy the convenience of instant cash loans at your fingertips. Our commitment to transparency and customer satisfaction sets us apart, ensuring a reliable and trustworthy lending experience. Experience the convenience of ZK Pesa for all your cash loan needs in Kenya.

💡 How to Tap into the Benefits?
1. Get started with your Kenya phone number and embark on your financial journey.
2. Fill in your basic details and let us take care of the rest.
3. Watch as your loan swiftly lands in your designated account.
4. Stay ahead of the game by making timely repayments and watching your credit score soar.

💪 Join Our League of Loan Warriors!
1. Age must be 18 years and above.
2. Employed/self-employed with a stable income.
3. Possess a mobile phone number with a mobile money wallet.
4. Must hold a national identity card.

💰 Insider Insights into Loans:
- Loan Amount: From Ksh 2,000 to Ksh 40,000
- Minimum Loan Repayment Period: 91 days including loan extension
- Maximum Loan Repayment Period: 365 days including loan extension
- Maximum APR: 36%
- Minimum APR: 12%
- Service Fee = Ksh 0

💡 Tips: Your credit limit and annual percentage rate (APR) are determined by our credit scoring system and will not exceed the APR range mentioned above. Your responsible repayment behavior will increase your credit limit and qualify you for lower interest rates.

📊 Dive into the Numbers with a Loan Example:
- Loan Amount = Ksh 5,000
- Loan Period = 91 days
- Service Fee = Ksh 0
- Annual Interest Rate = 18.1%
- Total Interest = Ksh 5,000 * 18.1% * 91/365 = Ksh 225
- Total Repayment Amount = Ksh 5,000 + Ksh 225= Ksh 5,225

🌐 More Than Just Loans - We're Your Financial Partner!
ZK Pesa is here to empower you on your financial journey. With no collateral needed, a seamless online process, lightning-fast approvals, and top-notch security, we've got you covered.

🔍 Questions? We're Here to Help!
Have queries about our loan process? Reach out anytime, and we'll guide you through every step.
Ready to take control of your finances? Let ZK Pesa lead the way! 🌟
Contact Us: +254 207905106
If you have any questions, please feel free to contact us. We will get back to you within three working days.
- Email: customersupport@zkpesa.com
- Office Hours: Monday to Saturday, 8 AM to 5 PM
- Official Website: https://web.zkpesa.com
- Address: 60 Peponi Rd, Nairobi, Kenya
💡ZK Pesa – Your Gateway to Financial Freedom!"""},

    {"provider_name": "Kashbean", "legal_name": "Creditarea Capital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://app-api.kashbean.co.ke/wap/agreement?name=Privacy_Policy&system=Kashbean", "raw_text": None, "local_file": None},
    {"provider_name": "Kashbean", "legal_name": "Creditarea Capital Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Kashbean is a digital lending platform that provides eligible customers in Kenya with convenient access to digital credit through a simple mobile application process.
Whether you need funds for business, education, medical expenses, household needs or other personal financial requirements, Kashbean offers a fast and transparent borrowing experience. Once approved, loan funds are disbursed directly to your registered M-PESA account.

Why Choose Kashbean loan？
• Fast digital loan application
• No collateral required
• Transparent loan terms and pricing
• Funds disbursed directly to your registered M-PESA account (upon approval)
• Flexible repayment through M-PESA
• Secure application and encrypted data protection

Loan Product:
• Loan Amount: up to Ksh 100,000
• Loan period: Customers can express a preference for loans up to 91 days, depending on the loan product offered and your eligibility
• Interest rate: Varies by loan product and will be disclosed before loan acceptance
• Representative APR: 425%-1600%
The applicable loan amount, interest, fees (if any), repayment date and total repayment amount will always be displayed before you accept your loan.

Representative Example
Loan amount: Ksh 10,000
Loan term: 61 days
Interest amount: Ksh 4,500
Total amount due: Ksh 10,000+Ksh 4,500= Ksh 14,500
Illustrative example only. Actual loan offers depend on eligibility.

Loan Eligibility
• Kenyan resident
• At least 18 years old
• Valid National ID
• Registered M-PESA account
• Meet our credit assessment
Loan approval is subject to credit assessment.

Loan Application Process
1.Register your account.
2.Complete identity verification.
3.Submit loan application.
4.Review loan offer.
5.Read and accept the Loan Agreement.
6.Upon approval, funds will be disbursed directly to registered M-PESA account.

Repayment
Repayments can be made securely through:
-Official M-PESA Paybill: 196184
-Account Number: Your registered M-PESA mobile number
-M-PESA Express (STK Push) through the Kashbean App

Security Reminder
For your protection, always make repayments using the official payment methods displayed in the Kashbean App.Kashbean will never request repayment through a personal M-PESA number, personal bank account or individual mobile wallet.If you receive different payment instructions, please do not make any payment and contact Customer Support immediately.

Data Security & Privacy
-Protecting your personal information is one of our highest priorities.
-Your information is encrypted during transmission and securely stored using industry-standard security measures.
-Your personal data will only be collected, processed and shared in accordance with our Privacy Policy, your Loan Agreement and applicable laws and regulations.
-Please review our Privacy Policy before registering.
-We do not sell your personal information.

Responsible Borrowing
Please borrow responsibly.
Borrow only what you need and ensure you can comfortably repay on time.
Before accepting a loan, you will always be shown:
-Loan Amount
-Amount to be Disbursed
-Interest and applicable charges
-Loan Term
-Repayment Date
-Total Repayment Amount
Timely repayment may improve your future borrowing eligibility, while late repayment may result in additional charges

Credit Reporting
Loan repayment information shall be shared with and updated at licensed Credit Reference Bureaus (CRBs) in accordance with applicable law and your Loan Agreement.

Customer Support
If you have any questions, feedback or complaints, please contact us:
Email:help@kashbean.co.ke
Call:0709267200"""},

    {"provider_name": "Zash Loan", "legal_name": "Zillions Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://app-api.zashloan.com/wap/agreement?name=Privacy_Policy&system=ZashLoan", "raw_text": None, "local_file": None},
    {"provider_name": "Zash Loan", "legal_name": "Zillions Credit Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Zash Loan is a digital lending platform designed to provide eligible customers in Kenya with convenient access to unsecured personal loans through a simple and secure mobile application process.

Customers are presented with the applicable loan amount, repayment amount, loan term and all applicable charges before deciding whether to accept a loan offer.

1.Product Information
• Loan Amount: Ksh500 to Ksh100,000 (subject to eligibility assessment).
• Loan Term: Customers may apply for loan products with repayment periods of up to 91 days, depending on the loan product offered and their eligibility.
• Representative APR Range: 274%–1,500% per annum, depending on the loan product, repayment period and applicable interest rate.
•Actual loan amounts, loan terms, applicable charges, repayment schedules and due dates are displayed within the App before customers decide whether to accept a loan offer.

Representative Example
• Loan Amount: Ksh 15,000
• Loan Term: 61 days
• Applicable Interest Rate for this Representative Example: 45%
• Total Interest Charge: Ksh 6,750
• Total Amount Due: Ksh 21,750
• Monthly Instalment (2 payments): Ksh 10,875 per instalment
No processing fees are charged unless otherwise disclosed before loan acceptance.
This representative example is provided for illustration purposes only. Actual loan amount, interest, applicable charges, repayment schedule and total repayment are determined based on the loan product offered and the customer's eligibility, and will be presented before loan acceptance.

Why Choose Zash？
• Paperless application
• No collateral required
• Fast digital application
• M-PESA disbursement upon approval
• Transparent loan information before acceptance
• Eligible customers may qualify for higher credit limits based on repayment history and continued eligibility.

2.Eligibility
• Be a Kenyan resident
• Be at least 18 years old
• Have a source of income
Meeting the eligibility criteria does not guarantee loan approval.

3.Repayment Methods
• M-PESA Paybill 733555 (Account Number: your registered mobile number)
• Zash Loan App using M-PESA Express
• Enter the repayment amount and confirm the transaction using your M-PESA PIN.
• Timely repayment may improve your eligibility for higher credit limits.
• Applicable repayment details are presented within the App before loan acceptance.

4.Privacy
Your personal information is processed securely in accordance with applicable Kenyan laws and our Privacy Policy. Where permitted or required by applicable law, your information may be shared with authorised service providers and licensed Credit Reference Bureaus (CRBs) for purposes permitted by law.

5.Consumer Disclosure
Customers are provided with sufficient time and information to review the applicable loan terms before deciding whether to accept or decline a loan offer.
Before any loan is disbursed, customers are presented with:
• Loan amount
• Applicable interest
• Fees (if any)
• Total repayment amount
• Repayment schedule
• Due date
Customers are under no obligation to accept a loan offer after reviewing the applicable loan terms, repayment schedule and total repayment amount.

6.CONTACT US!
Email: cs@zashloan.com
Website: www.zashloan.com
Call: +254709462300
WhatsApp:+254796848128"""},

    {"provider_name": "Flashpesa", "legal_name": "AMBUSH CAPITAL LIMITED", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://api.flashpesa.africa/privacypolicy.html", "raw_text": None, "local_file": None},
    {"provider_name": "Flashpesa", "legal_name": "AMBUSH CAPITAL LIMITED", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Join the fastest growing and responsible loan App in Kenya FlashPesa.
Access up to Ksh 80,000 on your smartphone — anytime and anywhere.

**The secure and reliable loan App FlashPesa App offers:
• Loans from Ksh 1000 to Ksh 80,000
• Loan terms: From 91 days to 365 days
• Origination fee: One-time fee From 150 Ksh - 600 Ksh
• Interest: 0%
• Maximum Annual Percentage Rate (APR) includes interests plus origination fee is 36%
• Loan request: Need to be at least 22 years old and in employment

**Example:
-If you apply amount Ksh 2000 with total 91days loan on 26th Mar 2021
-You will get the loan amount Ksh 2000 on 26th Mar 2021
-The due date is 24th Jun 2021 with amount Ksh 2000, Interest Ksh 0 and origination fees Ksh 150, due repayment is 2000+0+150=Ksh 2150
-Therefore, the total repayment should be Ksh 2150

How to apply a loan?
• Download FlashPesa App
• Create your account by filling in several quick questions
• Apply in seconds
• Receive your loan straight into your M-Pesa account
• Increase loan limit every time you repay on time

How to repay my loan?
• One click repayment button in FlashPesa App
• Or through M-Pesa to pay bill 911280 and your registered phone number as the account
• Also remember you can also make partial payments before the due date

As responsible mobile loan service provider in Kenya. We would like to share below reminders:
• Only borrow what you are able to repay
• Only borrow what you really need
• Only borrow when you are in employment
• Your credit history is very important so remember to repay on time to avoid repercussion
• We will only and always serve for your real demands

Privacy and Permissions:
When you download FlashPesa, we will ask for permissions to use the data on your phone in order to verify your identity and create a credit score. We encrypt the data you choose to share with us to protect your privacy. We take your privacy very seriously and promise to never share your information with third parties

Contact Us:
We’re always excited to hear from you! If you have any feedback, questions, or concerns.
Facebook Page: https://www.facebook.com/FlashPesa
E-mail: support@flashpesa.co.ke
Website: https://www.flashpesa.co.ke/
WhatsApp: +254769159474
Address: Kaka House, Manua Close, Westlands, Nairobi City, Kenya"""},

    {"provider_name": "Loop", "legal_name": "NCBA Bank", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://loop.co.ke/privacy_policy/", "raw_text": None, "local_file": None},

    {"provider_name": "FairKash+", "legal_name": "weshare limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://fbiz.fairkplus.cc/privacidad/privacidadProt.html", "raw_text": None, "local_file": None},
    {"provider_name": "FairKash+", "legal_name": "weshare limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Borrow up to KES 100,000 anytime with Fair Kash+! Our entirely digital and paperless application process approves you in minutes with no documentation needed. Apply once and withdraw cash as often as you need. We offer the most flexible credit line in Kenya.

Why Fair Kash+?
1.Up to KES 100,000 limit: Get approved once and borrow whenever needed. Hassle-free experience: Get approved once and re-borrow whenever you need in just a few taps. No more waiting or new applications required.
2. Fast cash: Get approved in minutes and money sent to your M-Pesa right away. We make borrowing simple.
3. Flexible repayments: Choose an installment plan or pay interest/principal whenever you want. Funds in your account instantly! Couldn't be easier!
4. Simple and safe: No hidden fees. Strict privacy and security regulated by Central Bank of Kenya. Trusted by 3 million+ Kenyans.

Loan Details:
- Loan Amount: KES 3,000 – KES 100,000
- Loan Term: 91 – 365 days
- Interest Rate: 15% – 36% (in compliance with applicable regulations)
- Service Fees: 2% – 10% (depending on loan terms and user profile)
- APR: 18%-48% (depending on loan terms and applicable fees)

Representative example:
Apply for a loan with period 120 days, Loan Principal is KES 10,000, Interest Fee Rate is 12%, Service Fee Rate is 6%.
- Loan Principal: KES 10,000 (120 days)
- Total Interest Fee: KES 1,200 (KES 10,000 * 12%)
- Total Service Fee: KES 600 (KES 10,000 * 6%)
- Total Fee Cost: KES 1,800 (KES 1,200 + KES 600)
- Repayment Amount: KES 11,800 (KES 10,000 + KES 1,200 + KES 600)

Who can Apply？
1. Any Kenyan citizen over 20 years old
2. Have a valid ID and mobile number

How to Get a Loan？
1. Download the Fair Kash+ app
2. Register with your phone number
3. Enter your basic info and loan request
4. Get approved instantly and receive money in minutes—deposited directly into your M-Pesa account

Join 1 million+ Kenyans enjoying freedom and flexibility with FairKash+. Apply once, borrow often, pay on your terms! Life's hard enough, we make your money simple. Get the app and experience better borrowing.

Privacy and Permissions
Fair Kash+ take your privacy very seriously and promise to never share your information with third parties.

Contact Us
We’re always here for you during business hours! If you have any feedback, questions, or concerns, please e-mail us at support@fairkplus.live
Message us on WhatsApp: +254 142430745
Address: Riverside Lane West, Riverside Building, Nairobi area, Kenya"""},

    {"provider_name": "Koro", "legal_name": "Zenka Digital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://koro.co.ke/koro-privacy-policy/", "raw_text": None, "local_file": None},
    {"provider_name": "Timiza", "legal_name": "ABSA", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.absabank.co.ke/data-privacy-statement/", "raw_text": None, "local_file": None},

    {"provider_name": "Letsgo", "legal_name": "Letshego Holdings Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.letsgo.letshego.com/privacy-policy.html", "raw_text": None, "local_file": None},
    {"provider_name": "Letsgo", "legal_name": "Letshego Holdings Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Letshego, your pan-Africa retail financial services partner now gives you an even better way to manage your finances with our new digital platform ‘LetsGo’, powered by Letshego.
Where we operate
The LetsGo app is accessible to customers in Letshego’s Africa footprint markets: Botswana, Eswatini, Ghana, Lesotho, Namibia, Nigeria, Tanzania and Uganda – find out more about us at www.letshego.com
Products we offer
• Loans*
• Payments*
Key digital platform features*
• Access Letshego products and services anytime, anywhere in just a few taps.
• View your latest Letshego balances, ie. loans and wallets.
• Apply digitally for Letshego products, including loans and wallets.
• Pay your bills, send and receive money through your LetsGo Pay Wallet
• Access Letshego’s expanding digital world of affordable ‘beyond banking’ solutions and support, including LetsGo Insure and LetsGo Wellbeing.
• LetsGoInsure is a simple and affordable way to protect you and your family.
• LetsGo Wellbeing is free for all LetsGo App users, providing health and wellness facts by registered doctors and health experts.
* - Features are subject to availability in your market guided by compliance.
How does it work?
- Simply download the app, select your country of residence and register!
Enjoy the benefits?
- Your data and personal information is secured through regulated world-class IT security platforms, using your ID number as a unique authentication reference.
- Check your personalised dashboard to view your active LetsGo products at a glance.
What if I need help?
• Contact our employees to walk you through the app using our live support features.
• Get answers to your questions using our detailed in-app FAQs .
• Chat with representatives to get immediate support to address your needs (Only available in Botswana).
Ready to get started? Let’s go! Install the DAS digital platform app now!
Here is an example of a Payroll Loan Offer
Loan amount: 50,000
Tenure: 72 months
Interest Rate: 24% per annum - 2% per month
Processing Fee: NIL
New customer onboarding Fee: NIL
VAT : n/a
Total Interest: 48,050.15
EMI: 1,456.22 (total monthly instal.)
APR: 24%
Loan amount is 50,000. Disbursed amount is 50,000.
Total loan repayment amount is 101,705.07
About Our Loans (the loan amounts, interest rate, tenure and currency vary for each Market, Products and Customer Risk Profile)
Loan amount range (min & max) – 250 to 500,000
Loan tenure range (min & max) – 3 months to 84 months
Maximum Annual Percentage Rate (APR) – 5.25% to 24% depending on credit profile
Lets Connect!
We want to hear from you! If you have any feedback, questions or concerns, message us within the app or email: digitalplatform@letshego.com.
Letshego is listed on the Botswana Stock Exchange and has been improving the lives of individuals and small business owners across Africa for over 20 years – why not join our #LetsGoNation and let us help you achieve your dreams, giving you the Power to Be & Do !"""},

    {"provider_name": "Advancepoa", "legal_name": "ZAGREUS LIMITED", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.advancepoa.com/privacy-policy", "raw_text": None, "local_file": None},
    {"provider_name": "Advancepoa", "legal_name": "ZAGREUS LIMITED", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """About AdvancePoa - Instant Personal Loans at Your Fingertips!
AdvancePoa is an instant personal loan app designed to provide you with essential longer term financial support. It helps you cover urgent expenses or seize business opportunities. With flexible repayment terms and a streamlined process, AdvancePoa keeps your finances on track without negatively affecting your credit rating.
💰 Loan Information
- Loan Amount: Ksh 5,000 – Ksh 400,000
- Repayment Period: 180 – 360 days
- Annual Percentage Rate (APR): 20.15% – 34.75%
💰 Representative Example of the Total Loan Cost
If you borrow KSh 100,000 with a repayment period of 360 days at an APR of 30%, here’s how the interest is calculated:
- Daily Interest Rate: 30% ÷ 365 ≈ 0.08%
- Total Interest for 360 Days: 100,000 × 0.08% × 360 ≈ KSh 28,800
- Total Repayment Amount: 100,000 + 28,800 = KSh 128,800
This is a sample calculation. Actual amounts will vary depending on the loan amount, repayment period, and APR. 😊
💵 Features & Benefits of AdvancePoa
1. Instant Loan Approval – Get funds transferred to your account within hours of approval.
2. Flexible Repayment Terms – Choose a repayment plan that aligns with your financial capacity.
3. No Collateral Required – Borrow with confidence; no assets or guarantors needed.
4. Transparent and Fair Rates – No hidden charges, just clear terms and affordable repayment plans.
5. Longer repayment period to ease your financial pressure
🚀 How to Get Your Instant Loan
- Download & Sign Up: Install the AdvancePoa app, register with your phone number, and create an account.
- Verify Your Details: Provide basic personal and employment information for eligibility checks.
- Request a Loan: Select your desired amount and submit your application.
- Receive Your Funds: Once approved, the money is deposited directly into your account within minutes.
🔐 Privacy and Data Protection
At AdvancePoa, your privacy is our priority. We collect data provided by you, such as personal details, to verify your identity and process your loan request. Rest assured, we protect your information and use it only for loan approval purposes. For details, see our Privacy Policy: https://www.advancepoa.com/privacy-policy
📞 Contact Us
Website: https://www.advancepoa.com/
Email: service@advancepoa.com
Address: Mayfair Center, Ralph Bunche Road, Nairobi, Kenya
Reach out anytime—we’re here to help! 😊"""},

    {"provider_name": "Pawacash", "legal_name": "Sokohela Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.pesaloan.com/asset/Pawa_Privacy_20260710.html", "raw_text": None, "local_file": None},
    {"provider_name": "Pawacash", "legal_name": "Sokohela Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Pawa Kash, your personal financial assistant!
18+ only
Apply for a loan through Pawa Kash, verify your identity, and enter simple personal information to receive funds in your M-PESA. A loan to kickstart a new life!

Pawa Kash Credit Features:
lOnline Processing: Say goodbye to painful and complicated procedures, everything is done through the app
lFast Approval: Get up to KES 50,000 with quick verification
lFunds in M-PESA: High approval rate, with loans transferred to M-PESA
lFlexible Repayment: Choose any repayment period from 91 to 180 days
lLow Interest Rates: Maximum annual interest rate of 24%

For example:
If you choose a loan limit of KES 5,000.00 with a period of 121 days
the interest rate of 18%,
total interest must be paid: KES 5,000.00 * (18%/365*121) =KES 298.00,
total payment of KES 5,298.00,
KES 5,000 (borrowed amount) + KES 298.00 (interest rate) =KES 5,298.00, (total payment)

Privacy Disclosure
Privacy & Permissions: Pawa Kash uses data from your phone, including your approximate location and device ID, to verify your identity and create a credit score, making it simple to get a personal loan. We encrypt the data you choose to share with us to protect your privacy. We take your privacy very seriously and promise to never share your information with third parties.

Data Security: Your data is safe with us. It is transmitted to us over a secure HTTPS connection and we will not share it with anyone except the lender without your consent.
Download the Pawa Kash mobile app now!
Contact us
Address: Malik Heights ,Along Ngong Road 7th Floor Nairobi ,Kenya
Phone: +254 793059320
Email: kashpawa069@gmail.com
Company website: https://www.pesaloan.com/
Privacy agreement address: https://www.pesaloan.com/asset/Pawa_Privacy_20260710.html
09:00 - 17:00 during workdays"""},

    {"provider_name": "Choice Sasa", "legal_name": "Choice Microfinance Bank Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://sasa.choice-bank.com/privacy-policy", "raw_text": None, "local_file": None},
    {"provider_name": "Tumacash", "legal_name": "TumaCash Internet Service Company", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.gettumax.com/policy.html", "raw_text": None, "local_file": None},
    {"provider_name": "Pesain", "legal_name": "Wabema Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.pesain.net/h5/secretPage.html", "raw_text": None, "local_file": None},
    {"provider_name": "Numida", "legal_name": "Numida Technologies Kenya Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://ke.numida.com/data-and-privacy-policy", "raw_text": None, "local_file": None},

    {"provider_name": "Boreleza", "legal_name": "BOREAS LIMITED", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://boreleza.com/privacy-policy", "raw_text": None, "local_file": None},
    {"provider_name": "Boreleza", "legal_name": "BOREAS LIMITED", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://boreleza.com/terms-of-use", "raw_text": None, "local_file": None},

    {"provider_name": "Helasasa", "legal_name": "LIGHTINGMICRO", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://sites.google.com/view/helasasaprivacy?pli=1&authuser=0", "raw_text": None, "local_file": None},
    {"provider_name": "Helasasa", "legal_name": "LIGHTINGMICRO", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """Hela Pesa provides personal loans for a minimum repayment period of between 90 days and a maximum repayment period of 12 months.
Personal loans range from Ksh 500 to Ksh 200,000
Tenure: 3 months to 2 years. Interest rates range from 2%-15% depending on the risk profile of the customer.
Example: Borrow Ksh 1,000 at APR of 60% (per annum) with tenure of 3 Months
Interest = Ksh. 1000 * 60%/ 12 * 3 = Ksh. 150
Total amount disbursed will be Ksh 1000
Total amount to repay will be Ksh 1150 (Ksh 1000 + Ksh 150)
Three monthly payments of Ksh 383.33

Personal loan rates and fees
Loan Amounts: Ksh 500 to Ksh 200,000
APR/Interest rate: 30%-180% (Monthly interest: 2.5% to 15%)
Processing fee: None
Onboarding fee: None
TERMS: We offer loans that are payable for a minimum of 90 days and a maximum of 2 years.

HELA PESA is a loan service application that is specially designed to give clients access to quick, convenient and affordable mobile loan products anywhere and anytime. Loans are disbursed and paid using the mobile money transfer services like MPESA. To access the service, download the app, complete registration process and wait for account activation. Once account is activated, loan limit is availed which is the loan ceiling.
This limit is pegged on the ability of the client to repay that is derived from the financial and other data provided by the client. You can also switch between modules in order to activate and access other products like salary loans, mobile loans among others The app also provides a convenient platform for loan repayments enabled by the integrated mobile money payment service. Data and transactions are highly secured to provide clients with a secure business environment. This application is a product of Hela Capital Ltd that prides in providing convenient financial services, anywhere and anytime."""},

    {"provider_name": "Zuri Cash", "legal_name": "Whitepath Company Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://support.zuricash.live/privacy_privacy/", "raw_text": None, "local_file": None},
    {"provider_name": "Ganji Pesa", "legal_name": "GANJI TECH LIMITED", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.ganjipesa.com/privacy.html", "raw_text": None, "local_file": None},
    {"provider_name": "Ganji Pesa", "legal_name": "GANJI TECH LIMITED", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.ganjipesa.com/service.html", "raw_text": None, "local_file": None},

    {"provider_name": "Chapaa Loan", "legal_name": "Chapaa Loan", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://sites.google.com/view/chapaa-chapaa", "raw_text": None, "local_file": None},
    {"provider_name": "Chapaa Loan", "legal_name": "Chapaa Loan", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """*Chapaa Loan was founded to offer better financial service that is simple, fully digital and no paperwork to non-bank people with the world's advanced technologies.
*Loan Product
1. Loans Amount: Ksh 2,500 to Ksh 50,000
2. Loan Tenure: 91 days - 365 days
3. APR: Minimum 24% to Maximum 50%
Example for Better Understand: If the loan amount is Ksh 5000 and the APR is 31% with a tenure of 91days
Interest = 5000 Ksh * 31% / 365 * 91 = 386Ksh
You will get Ksh 5000 and should repay Ksh 5,386 (5000 + 386) after 91 days.

* Loan Process
1. Install app from the Play Store.
2. Create a Chapaa Loan account.
3. Choose the amount you would like to apply for.
4. Fill in your basic information and submit.
5. Receive cash straightly into your M-pesa account.

*Permission and Privacy
When installing Chapaa Loan, we will require you to authorize us to access your mobile information to verify your identity and evaluate your credit eligibility. Your data is protected by the highest security standards."""},

    {"provider_name": "Raha Pesa", "legal_name": "Mkulimapay Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.mkulimapay.co.ke/rahapesa-privacy-policy/", "raw_text": None, "local_file": None},
    {"provider_name": "Raha Pesa", "legal_name": "Mkulimapay Credit Limited", "document_type": "loan_product_information", "source_type": "google_play_store", "source_url": "Google Play Store", "local_file": None, "raw_text": """🔥 Need Urgent Cash? Raha Pesa is Your Instant Solution!
💸 Get Cash Fast, No Stress!
✅ Loan Amount: Ksh 5,000 – Ksh 500,000
✅ Flexible Repayment: 91 Days to 180 Days
✅ Maximum APR: 34%
✅ Minimum APR: 16%
✅ Max Limit: Ksh 500,000
✅ No Documents Needed! • No Collateral!
📌 Example Loan Breakdown:
Borrow Ksh 5,000 at 18.25% APR for 91 days
Interest Fee: Ksh 225
Total Repayment: Ksh 5,000+ Ksh 225 (Principal + Interest)=Ksh 5,225
⭐ Why Choose Raha Pesa?
✔ 100% Digital – Apply, Receive, Repay – All on Your Phone!
✔ Instant Disbursement → Straight to M-Pesa! 💰
✔ 24/7 Access – Need Cash at Night? No Problem! 🌙
✔ Smart Borrowing = Higher Limits – Build Your Credit!
✔ Extended Repayment? We Adjust to Fit Your Needs! ⏳
✔ Countrywide Coverage – Kenyans Everywhere Can Apply! 🇰🇪
✅ Who Can Get a Loan?
Kenyan Citizen
Age 18–55
Regular Income (Even Small Business & Casual Work!)
🔐 Your Data & Money Are Safe!
✔ Bank-Level Encryption – No Fraud Risk!
✔ No Data Selling – We Never Share Your Info!
✔ Licensed & Regulated – Reliable Instant Loan!
📩 Need Help? Talk to Us!
📧 Email: cs.rahapesa.android@mkulimapay.co.ke
📞 Call/WhatsApp: +254 207905941"""},

    {"provider_name": "Bayes", "legal_name": "Pi Capital Ltd", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.bayes.co.ke/terms-and-conditions", "raw_text": None, "local_file": None},
    {"provider_name": "Bayes", "legal_name": "Pi Capital Ltd", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.bayes.co.ke/privacy-policy", "raw_text": None, "local_file": None},

    {"provider_name": "Asapkash", "legal_name": "ZEROX TECHNOLOGY COMPANY LIMITED", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.asapkash.co.ke/privacyPolicy", "raw_text": None, "local_file": None},

    {"provider_name": "Kechita Capital", "legal_name": "Kechita Capital Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://kechitacapital.co.ke/terms-and-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Kechita Capital", "legal_name": "Kechita Capital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://kechitacapital.co.ke/privacy-policy/", "raw_text": None, "local_file": None},

    {"provider_name": "Pato Capital", "legal_name": "Pato Capital Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://patocapital.com/terms/", "raw_text": None, "local_file": None},
    {"provider_name": "Pato Capital", "legal_name": "Pato Capital Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://patocapital.com/privacy/", "raw_text": None, "local_file": None},

    {"provider_name": "Tick Credit", "legal_name": "Tick Credit", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.tickconsulting.co.ke/terms-and-conditions/index.htm", "raw_text": None, "local_file": None},

    {"provider_name": "iLoan", "legal_name": "Easy Asset Management Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://iloan.co.ke/terms-and-conditions", "raw_text": None, "local_file": None},
    {"provider_name": "iLoan", "legal_name": "Easy Asset Management Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://iloan.co.ke/privacy-policy", "raw_text": None, "local_file": None},

    {"provider_name": "Asapcredit", "legal_name": "Asap Credit Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://asapcredit.africa/terms-and-conditions", "raw_text": None, "local_file": None},
    {"provider_name": "Asapcredit", "legal_name": "Asap Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://asapcredit.africa/privacy-policy", "raw_text": None, "local_file": None},

    {"provider_name": "Kopokopo", "legal_name": "Kopo Kopo Inc.", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://app.kopokopo.com/privacy", "raw_text": None, "local_file": None},
    {"provider_name": "Kopokopo", "legal_name": "Kopo Kopo Inc.", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://kopokopo.co.ke/terms-conditions/", "raw_text": None, "local_file": None},

    {"provider_name": "Coop Bank", "legal_name": "Coop Bank", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.co-opbank.co.ke/all_terms_conditions/digital-lending-terms-conditions/", "raw_text": None, "local_file": None},

    {"provider_name": "Tazu Credit", "legal_name": "Tazu Credit Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://tazugroup.com/legal/terms", "raw_text": None, "local_file": None},
    {"provider_name": "Tazu Credit", "legal_name": "Tazu Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://tazugroup.com/legal/privacy", "raw_text": None, "local_file": None},

    {"provider_name": "Tanir Credit", "legal_name": "Tanir Credit & Accounting Services Limited", "document_type": "loan_agreement", "source_type": "official_website", "source_url": "https://tanirltd.co.ke/loan-agreement-terms", "raw_text": None, "local_file": None},
    {"provider_name": "Tanir Credit", "legal_name": "Tanir Credit & Accounting Services Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://tanirltd.co.ke/privacy", "raw_text": None, "local_file": None},
    {"provider_name": "Tanir Credit", "legal_name": "Tanir Credit & Accounting Services Limited", "document_type": "terms_of_use", "source_type": "official_website", "source_url": "https://tanirltd.co.ke/terms-use", "raw_text": None, "local_file": None},

    {"provider_name": "Premier Credit", "legal_name": "Premier Credit Limited", "document_type": "terms_and_conditions", "source_type": "official_website", "source_url": "https://www.premierkenya.co.ke/quick-links/terms-and-conditions/", "raw_text": None, "local_file": None},
    {"provider_name": "Premier Credit", "legal_name": "Premier Credit Limited", "document_type": "privacy_policy", "source_type": "official_website", "source_url": "https://www.premierkenya.co.ke/quick-links/privacy-policy/", "raw_text": None, "local_file": None}
]

In [ ]:
# 2. Directory Structure Setup
def setup_directories():
    base_dirs = ["dataset/raw", "dataset/extracted", "dataset/metadata"]
    for d in base_dirs:
        os.makedirs(d, exist_ok=True)
    print("Directory structure initialized.")

In [ ]:
# 3. HTML Cleaning & Extraction
def clean_html_to_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    for element in soup(["nav", "header", "footer", "script", "style", "aside", "button", "svg"]):
        element.decompose()
    return soup.get_text(separator="\n", strip=True)

In [ ]:
# 4. Main Async Scraping & Processing Engine
async def process_dataset():
    setup_directories()
    documents_data = []
    source_registry = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        for source in SOURCES:
            doc_id = f"DOC_{uuid.uuid4().hex[:8].upper()}"
            provider_folder = f"dataset/raw/{source['provider_name'].lower().replace(' ', '_')}"
            os.makedirs(provider_folder, exist_ok=True)

            extracted_text = ""
            original_file_name = source['local_file'] or f"{source['document_type']}.html"
            date_accessed = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            document_title = f"{source['provider_name']} {source['document_type'].replace('_', ' ').title()}"

            print(f"Processing: {document_title}...")

            # ROUTE A: Live Web Scraping
            if source['source_type'] == "official_website" and source['source_url']:
                try:
                    page = await context.new_page()
                    await page.goto(source['source_url'], wait_until="networkidle", timeout=30000)
                    html = await page.content()

                    raw_path = os.path.join(provider_folder, original_file_name)
                    with open(raw_path, "w", encoding="utf-8") as f:
                        f.write(html)

                    extracted_text = clean_html_to_text(html)
                    await page.close()
                except Exception as e:
                    print(f"  -> Error scraping {source['source_url']}: {e}")
                    extracted_text = "ERROR: Failed to scrape"

            # ROUTE B: Provided Text (Play Store or Offline PDFs)
            else:
                extracted_text = source['raw_text']
                raw_path = os.path.join(provider_folder, f"{source['document_type']}.txt")
                original_file_name = f"{source['document_type']}.txt"
                with open(raw_path, "w", encoding="utf-8") as f:
                    f.write(extracted_text)

            # Append to Documents Dataset
            documents_data.append({
                "document_id": doc_id,
                "provider_name": source['provider_name'],
                "legal_name": source['legal_name'],
                "document_type": source['document_type'],
                "source_type": source['source_type'],
                "source_url": source['source_url'],
                "date_accessed": date_accessed,
                "document_title": document_title,
                "full_extracted_text": extracted_text,
                "original_file_name": original_file_name
            })

            # Append to Source Registry
            source_registry.append({
                "document_id": doc_id,
                "provider_name": source['provider_name'],
                "source_url": source['source_url'],
                "local_path": raw_path,
                "status": "Success" if "ERROR" not in extracted_text else "Failed"
            })

        await browser.close()
    return documents_data, source_registry

In [ ]:
# 5. Export to CSVs

if __name__ == "__main__":
    documents_data, source_registry = await process_dataset()

    docs_df = pd.DataFrame(documents_data)
    docs_df.to_csv("dataset/extracted/documents.csv", index=False, quoting=csv.QUOTE_ALL)

    registry_df = pd.DataFrame(source_registry)
    registry_df.to_csv("dataset/metadata/source_registry.csv", index=False)

    print("\nDataset generation complete!")
    print(f"Total documents processed: {len(docs_df)}")